# 13 — Tonalités du DUNUN (ouvert / fermé)

Le jembé a 3 frappes (02). Le **dunun**, joué à la baguette, en a typiquement 2 : **ouvert** (résonant, grave, fort) et **fermé/étouffé** (main posée sur la peau, plus mat et faible). On le vérifie par clustering sur les frappes de dunun isolées.

Séparation plus nette que le jembé (silhouette ~0.39) : les deux tonalités sont bien distinctes.

## 0. Imports & frappes de dunun (avec cache)

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap, djembe_utils as dj

CACHE = 'output/all_dunun_strokes_typed.csv'
c = dj.list_corpus()
if os.path.exists(CACHE):
    dn = pd.read_csv(CACHE); print('Cache :', len(dn), 'frappes de dunun')
else:
    rows = []
    for _, r in c[c['role']=='dunun'].iterrows():
        y = dj.load_track(r['piece'], r['take'], r['code'])
        f = dj.fast_stroke_features(y, dj.load_onsets(r['piece'], r['take'], r['code'])['Time'].values)
        f['piece'], f['take'], f['code'] = r['piece'], r['take'], r['code']; rows.append(f)
    dn = pd.concat(rows, ignore_index=True); dn.to_csv(CACHE, index=False)
    print('Construit :', len(dn), 'frappes')

## 1. Clustering en 2 tonalités
**ouvert** = fort + grave (max énergie) ; **fermé** = plus faible et mat.

In [ ]:
X = StandardScaler().fit_transform(dn[dj.FAST_FEATS])
for k in (2,3):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f'k={k}: silhouette={silhouette_score(X, lab, sample_size=4000, random_state=0):.3f}')
dn['cl'] = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
g = dn.groupby('cl')['rms'].mean()
names = {g.idxmax():'ouvert', g.idxmin():'fermé'}
dn['tonalite'] = dn['cl'].map(names)
print()
print(dn.groupby('tonalite').agg(low_ratio=('low_ratio','mean'), centroid=('centroid','mean'),
      rms=('rms','mean'), n=('tonalite','size')).round(3))

## 2. Écouter les deux tonalités

In [ ]:
_tr = {}
def get(piece, take, code):
    k=(piece,int(take),code)
    if k not in _tr: _tr[k]=dj.load_track(piece,int(take),code)
    return _tr[k]
def ecouter(ton, n=14, dur=0.28):
    s = dn[dn['tonalite']==ton].sample(min(n,(dn['tonalite']==ton).sum()), random_state=1)
    clips=[get(r['piece'],r['take'],r['code'])[int(r['time']*dj.SR):int(r['time']*dj.SR)+int(dur*dj.SR)] for _,r in s.iterrows()]
    print(f'>>> {ton}'); display(Audio(np.concatenate(clips), rate=dj.SR))
for t in ['ouvert','fermé']: ecouter(t)

## 3. Carte + proportion par rythme

In [ ]:
emb = umap.UMAP(n_neighbors=15, min_dist=0.15, random_state=42).fit_transform(X)
dn['ux'], dn['uy'] = emb[:,0], emb[:,1]
COL = {'ouvert':'#2b6cff', 'fermé':'#7fb0ff'}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for t, cc in COL.items():
    m = dn['tonalite']==t; ax[0].scatter(dn.loc[m,'ux'], dn.loc[m,'uy'], s=6, c=cc, alpha=.5, label=t)
ax[0].legend(); ax[0].set_title('Frappes de dunun (UMAP)')
pd.crosstab(dn['piece'], dn['tonalite'], normalize='index').plot.bar(stacked=True, color=[COL['fermé'],COL['ouvert']], ax=ax[1])
ax[1].set_title('Proportion ouvert/fermé par rythme'); ax[1].set_xticklabels(dj.PIECES, rotation=0)
plt.tight_layout(); plt.savefig('output/13_dunun_tonalites.png', dpi=130); plt.show()

## Bilan
- Le dunun a **2 tonalités** bien séparées (ouvert résonant / fermé étouffé).
- Avec le jembé (b/t/s), on a désormais un **vocabulaire de frappes par instrument** complet → utile pour enrichir les partitions et la génération.
